<a href="https://colab.research.google.com/github/KamilAkarsu/DoktoraDeneyleri/blob/main/Exp_2_13_MURA_and_Modern_CNN(regnet_y_3_2gf).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Hücre 1: Kütüphane İçe Aktarımları ve Veri Ortamının Hazırlanması

# ==============================================================================
# TEMEL KÜTÜPHANELERİN İÇE AKTARILMASI
# ==============================================================================
import os           # İşletim sistemi işlemleri, klasör/dosya yolu yönetimi
import time         # Eğitim sürelerinin ve çıkarım (inference) hızlarının hesaplanması
import zipfile      # Sıkıştırılmış veri setinin (MURA) yerel diske çıkartılması

# ==============================================================================
# PYTORCH VE DERİN ÖĞRENME MODÜLLERİ
# ==============================================================================
import torch        # Ana derin öğrenme çerçevesi
import torch.nn as nn # Sinir ağı katmanları ve kayıp (loss) fonksiyonları
import torch.optim as optim # Optimizasyon algoritmaları (AdamW vb. ve Öğrenme Oranı Planlayıcıları)
from torch.utils.data import Dataset, DataLoader # Özelleştirilmiş veri seti ve toplu veri yükleyici sınıfları
from torchvision import transforms, models # Dinamik veri artırımı (augmentation) ve ön-eğitimli jenerik modeller

# ==============================================================================
# VERİ İŞLEME VE GÖRSELLEŞTİRME MODÜLLERİ
# ==============================================================================
from PIL import Image # Tıbbi görüntülerin (Röntgen) diskten okunması
import numpy as np  # Vektörel matris işlemleri
import pandas as pd # Eğitim geçmişinin ve sonuçların tablo halinde kaydedilmesi (CSV)
from tqdm import tqdm # Eğitim ve doğrulama döngülerinde iterasyon ilerleyişinin görselleştirilmesi

# ==============================================================================
# PERFORMANS METRİKLERİ (SCIKIT-LEARN)
# ==============================================================================
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, cohen_kappa_score, confusion_matrix,
                             roc_auc_score, average_precision_score,
                             mean_absolute_error, mean_squared_error)

# ==============================================================================
# ÇALIŞMA ORTAMI VE VERİ BAZI HAZIRLIKLARI
# ==============================================================================
# 1. Google Drive Bağlantısı: Ağırlıkların ve grafiklerin kaydedileceği kalıcı disk
from google.colab import drive
drive.mount('/content/drive')

# 2. Veri Setinin Yerel Diske Çıkartılması
# Bulut depolamadan (Drive) doğrudan veri okumak I/O darboğazına (bottleneck) yol açar.
# GPU'nun veriyi beklemesini engellemek için veri seti geçici, hızlı yerel belleğe alınır.
DRIVE_ZIP_YOLU = '/content/drive/MyDrive/Doktora/Verisetleri_tik/Islenmis_MURA.zip'
YEREL_VERI_KLASORU = '/content/dataset'

if not os.path.exists(YEREL_VERI_KLASORU):
    print(f"Veri seti yerel diske çıkartılıyor: {DRIVE_ZIP_YOLU} ...")
    with zipfile.ZipFile(DRIVE_ZIP_YOLU, 'r') as zip_ref:
        zip_ref.extractall(YEREL_VERI_KLASORU)
    print("Çıkartma işlemi tamamlandı.")
else:
    print("Veri seti yerel diskte zaten mevcut.")

# ==============================================================================
# DONANIM HIZLANDIRICI ATAMASI
# ==============================================================================
# A100 GPU Kontrolü: Tensor işlemlerinin CPU yerine CUDA mimarisinde yürütülmesi için
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Aktif İşlem Birimi: {device}")

Mounted at /content/drive
Veri seti yerel diske çıkartılıyor: /content/drive/MyDrive/Doktora/Verisetleri_tik/Islenmis_MURA.zip ...
Çıkartma işlemi tamamlandı.
Aktif İşlem Birimi: cuda


hücre 1 sözde kod

In [2]:
# ==============================================================================
# HÜCRE 2: İZOLE HİPERPARAMETRE KONFİGÜRASYONU (EXP 2.13 - REGNET)
# ==============================================================================
CONFIG = {
    # Deney ismi RegNet olarak güncellendi
    "experiment_name": "Exp 2.13: MURA and Modern CNN(regnet_y_3_2gf)",

    # Model mimarisi RegNet olarak belirlendi
    "model_architecture": "regnet_y_3_2gf",

    # --- AŞAĞIDAKİ TÜM PARAMETRELER DİĞER DENEYLERLE %100 AYNIDIR ---
    "target_image_size": (224, 224),
    "batch_size": 32,
    "learning_rate": 5e-5,
    "num_epochs": 50,
    "weight_decay": 5e-3,
    "early_stopping_patience": 12,
    "use_mixup": False,
    "mixup_alpha": 0.2,
    "num_workers": 4,

    "augmentations": {
        "random_rotation_degrees": 15,
        "horizontal_flip_prob": 0.5,
        "color_jitter_brightness": 0.2,
        "color_jitter_contrast": 0.2
    }
}

hücre 2 sözde kod

In [3]:
#Hücre 3: Jenerik Veri Yükleyici ve Dinamik Artırma

import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# ==============================================================================
# ÖZELLEŞTİRİLMİŞ TIBBİ GÖRÜNTÜ VERİ KÜMESİ SINIFI
# ==============================================================================
# PyTorch'un standart Dataset sınıfından türetilen bu yapı, MURA veri setinin
# karmaşık klasör hiyerarşisini dinamik olarak tarayarak görüntüleri ve etiketleri eşleştirir.
class JenerikMedikalDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        # MURA veri setinin doğası gereği etiketler klasör/dosya isimlerinde gizlidir.
        # 'positive' = 1 (Kırık/Anormal), 'negative' = 0 (Sağlıklı/Normal)
        for root, dirs, files in os.walk(root_dir):
            for img_name in files:
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    tam_yol = os.path.join(root, img_name)
                    # Sınıflandırma problemi için etiket çıkarma (Label Extraction)
                    if 'positive' in tam_yol.lower():
                        self.image_paths.append(tam_yol)
                        self.labels.append(1)
                    elif 'negative' in tam_yol.lower():
                        self.image_paths.append(tam_yol)
                        self.labels.append(0)

    # Veri setindeki toplam örnek sayısını döndürür (Epoch iterasyonları için gereklidir)
    def __len__(self):
        return len(self.image_paths)

    # Verilen indeksteki görüntüyü diskten anlık olarak (on-the-fly) çeker.
    # Bu yöntem, tüm veri setini RAM'e yükleyip sistemi çökertmekten kurtarır.
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        # Eğer augmentasyon veya normalizasyon tanımlanmışsa görüntüyü deforme eder
        if self.transform:
            image = self.transform(image)
        return image, label

# =====================================================================
# DİNAMİK VERİ ARTIRMA (ON-THE-FLY AUGMENTATION)
# =====================================================================
# Modelin ezberlemesini (overfitting) önlemek için eğitim verilerine her iterasyonda
# rastgele dönüşümler uygulanır. Değerler izole CONFIG sözlüğünden çekilir.
train_transforms = transforms.Compose([
    # Modellerin (ResNet vb.) beklediği standart girdi matris boyutuna ayarlama
    transforms.Resize(CONFIG["target_image_size"]),

    # 1. Rotasyon: Tıbbi çekim hatalarını ve açı farklarını simüle eder
    transforms.RandomRotation(CONFIG["augmentations"]["random_rotation_degrees"]),

    # 2. Çevirme: Anatomik simetriyi kullanarak veri havuzunu genişletir
    transforms.RandomHorizontalFlip(p=CONFIG["augmentations"]["horizontal_flip_prob"]),

    # 3. Parlaklık/Kontrast: Farklı röntgen cihazlarının X-ışını doz farklarını simüle eder
    transforms.ColorJitter(
        brightness=CONFIG["augmentations"]["color_jitter_brightness"],
        contrast=CONFIG["augmentations"]["color_jitter_contrast"]
    ),

    # Görüntüyü PyTorch tensörüne çevirir ve piksel değerlerini [0, 1] aralığına çeker
    transforms.ToTensor(),

    # ImageNet Ön-Eğitimli (Pre-trained) modellerin beklediği standart normalizasyon değerleri.
    # Modelin çok daha hızlı ve stabil yakınsamasını (convergence) sağlar.
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Doğrulama/Test setleri modelin asıl performansını ölçeceği için deformasyona uğratılmaz (Augmentation uygulanmaz)
# Sadece tensör dönüşümü ve ImageNet normalizasyonu yapılır.
val_transforms = transforms.Compose([
    transforms.Resize(CONFIG["target_image_size"]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Klasör yolları (MURA v1.1 yapısına göre)
TRAIN_DIR = os.path.join(YEREL_VERI_KLASORU, 'MURA-v1.1', 'train')
VAL_DIR = os.path.join(YEREL_VERI_KLASORU, 'MURA-v1.1', 'valid')

# Veri seti nesnelerinin oluşturulması
train_dataset = JenerikMedikalDataset(root_dir=TRAIN_DIR, transform=train_transforms)
val_dataset = JenerikMedikalDataset(root_dir=VAL_DIR, transform=val_transforms)

# DataLoader: Verileri diskten GPU'ya belirlenen yığınlar (batch) halinde taşıyan köprü yapısı.
# pin_memory=True: CPU RAM'inden GPU VRAM'ine veri transferini hızlandırır.
train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

print(f"Eğitim Verisi: {len(train_dataset)} | Doğrulama Verisi: {len(val_dataset)}")

# ==============================================================================
# MIXUP REGÜLARİZASYONU (İSTEĞE BAĞLI / ABLASYON)
# ==============================================================================
# İki farklı görüntüyü ve etiketini Beta dağılımı üzerinden şeffaf bir şekilde üst üste bindirerek
# modelin kesin karar sınırlarını yumuşatmayı hedefleyen ileri düzey veri artırımı tekniği.
def mixup_data(x, y, alpha=CONFIG["mixup_alpha"]):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)

    # İki görüntünün piksellerini lambda (lam) oranında karıştırır
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

# Mixup işlemi uygulandığında standart kayıp fonksiyonunun da bu karışıma göre güncellenmesi gerekir.
def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

Eğitim Verisi: 36808 | Doğrulama Verisi: 3197


hücre 3 sözde kod

In [4]:
# ==============================================================================
# HÜCRE 4: KAPSAMLI JENERİK MODEL OLUŞTURUCU (REGNET EKLENDİ)
# ==============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

def jenerik_model_olustur(model_adi, num_classes=2, dropout_rate=0.5):
    print(f"[{model_adi}] mimarisi ImageNet ağırlıklarıyla yükleniyor... (Dropout: {dropout_rate})")

    # --- RESNET AİLESİ ---
    if model_adi == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        model.fc = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.fc.in_features, num_classes))
    elif model_adi == "resnet101":
        model = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V2)
        model.fc = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.fc.in_features, num_classes))

    # --- DENSENET AİLESİ ---
    elif model_adi == "densenet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        model.classifier = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.classifier.in_features, num_classes))
    elif model_adi == "densenet169":
        model = models.densenet169(weights=models.DenseNet169_Weights.IMAGENET1K_V1)
        model.classifier = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.classifier.in_features, num_classes))
    elif model_adi == "densenet201":
        model = models.densenet201(weights=models.DenseNet201_Weights.IMAGENET1K_V1)
        model.classifier = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.classifier.in_features, num_classes))

    # --- EFFICIENTNET AİLESİ ---
    elif model_adi == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        model.classifier[1] = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.classifier[1].in_features, num_classes))
    elif model_adi == "efficientnet_b4":
        model = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.DEFAULT)
        model.classifier[1] = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.classifier[1].in_features, num_classes))
    elif model_adi == "efficientnet_v2_s":
        model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
        model.classifier[1] = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.classifier[1].in_features, num_classes))

    # --- CONVNEXT AİLESİ ---
    elif model_adi == "convnext_base":
        model = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)
        model.classifier[2] = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.classifier[2].in_features, num_classes))

    # --- REGNET AİLESİ ---
    elif model_adi == "regnet_y_3_2gf":
        model = models.regnet_y_3_2gf(weights=models.RegNet_Y_3_2GF_Weights.IMAGENET1K_V2)
        model.fc = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.fc.in_features, num_classes))

    # --- MOBILENET ---
    elif model_adi == "mobilenet_v3_large":
        model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        model.classifier[3] = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.classifier[3].in_features, num_classes))

    # --- VGG AİLESİ ---
    elif model_adi == "vgg16_bn":
        model = models.vgg16_bn(weights=models.VGG16_BN_Weights.IMAGENET1K_V1)
        model.classifier[6] = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.classifier[6].in_features, num_classes))
    elif model_adi == "vgg19_bn":
        model = models.vgg19_bn(weights=models.VGG19_BN_Weights.IMAGENET1K_V1)
        model.classifier[6] = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.classifier[6].in_features, num_classes))

    # --- ALEXNET ---
    elif model_adi == "alexnet":
        model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
        model.classifier[6] = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.classifier[6].in_features, num_classes))

    # ==========================================================
    # --- VISION TRANSFORMERS (ViT & SWIN) ---
    # ==========================================================
    elif model_adi == "vit_b_16":
        model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
        model.heads.head = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.heads.head.in_features, num_classes))

    else:
        raise ValueError(f"HATA: '{model_adi}' tanımlı bir model değil. Lütfen CONFIG sözlüğünü kontrol edin.")

    # ==========================================================
    # JENERİK KATMAN DONDURMA (TRANSFER LEARNING STRATEJİSİ)
    # ==========================================================
    dondurulan_katman_sayisi = 0
    acik_katman_sayisi = 0

    for name, param in model.named_parameters():
        # "trunk_output.block4", RegNet mimarilerinde ağın en son ana özellik çıkarma bloğudur.
        if any(keyword in name for keyword in ["layer3", "layer4", "denseblock4", "features.7", "features.8", "trunk_output.block4", "fc", "classifier", "head"]):
            param.requires_grad = True
            acik_katman_sayisi += 1
        else:
            param.requires_grad = False
            dondurulan_katman_sayisi += 1

    print(f"Transfer Learning Stratejisi: {dondurulan_katman_sayisi} katman donduruldu, {acik_katman_sayisi} katman eğitiliyor.")

    return model.to(device)

# Modeli başlatma
model = jenerik_model_olustur(CONFIG["model_architecture"], dropout_rate=0.5)

# ==========================================================
# FARKILAŞTIRILMIŞ ÖĞRENME ORANI (DISCRIMINATIVE FINE-TUNING)
# ==========================================================
head_params = []
backbone_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue

    if any(keyword in name for keyword in ["fc", "classifier", "head"]):
        head_params.append(param)
    else:
        backbone_params.append(param)

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': CONFIG["learning_rate"] / 10},
    {'params': head_params, 'lr': CONFIG["learning_rate"]}
], weight_decay=CONFIG["weight_decay"])

# ==========================================================
# AĞIRLIKLI KAYIP FONKSİYONU (CLASS IMBALANCE HANDLING)
# ==========================================================
class_weights = torch.tensor([1.0, 1.5]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

print(f"Model başarıyla GPU'ya ({device}) taşındı ve eğitime hazır.")

[regnet_y_3_2gf] mimarisi ImageNet ağırlıklarıyla yükleniyor... (Dropout: 0.5)
Downloading: "https://download.pytorch.org/models/regnet_y_3_2gf-9180c971.pth" to /root/.cache/torch/hub/checkpoints/regnet_y_3_2gf-9180c971.pth


100%|██████████| 74.6M/74.6M [00:00<00:00, 242MB/s]


Transfer Learning Stratejisi: 192 katman donduruldu, 98 katman eğitiliyor.
Model başarıyla GPU'ya (cuda) taşındı ve eğitime hazır.


hücre 4 sözde kod

In [5]:
#Hücre 5: Kapsamlı Başarım Metrikleri Hesaplayıcısı

# ==============================================================================
# KAPSAMLI BAŞARIM METRİKLERİ HESAPLAYICISI
# ==============================================================================
# Modelin ürettiği tahminleri (predictions) ve olasılık skorlarını (probabilities)
# gerçek etiketlerle (ground truth) karşılaştırarak, tıbbi tanı sistemleri için
# literatürde kabul gören tüm değerlendirme ölçütlerini tek seferde hesaplar.

def kapsamli_metrikleri_hesapla(y_true, y_pred, y_scores):
    # Karmaşıklık Matrisi (Confusion Matrix) hesaplaması
    # tn (True Negative), fp (False Positive), fn (False Negative), tp (True Positive)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0,0,0,0)

    # Scikit-learn fonksiyonları kullanılarak metriklerin sözlük yapısında toplanması
    metrikler = {
        # Genel doğruluk oranı (Sınıf dengesizliği olan durumlarda tek başına yanıltıcı olabilir)
        "Accuracy": accuracy_score(y_true, y_pred),

        # Kesinlik: Modelin "Kırık (1)" dediklerinin ne kadarının gerçekten kırık olduğu
        "Precision": precision_score(y_true, y_pred, zero_division=0),

        # Duyarlılık (Recall/Sensitivity): Gerçekte "Kırık" olanların ne kadarının model tarafından bulunabildiği
        "Recall_Sensitivity": recall_score(y_true, y_pred, zero_division=0),

        # Özgüllük (Specificity): Gerçekte "Sağlıklı" olanların ne kadarının doğru tespit edildiği
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,

        # F1-Skoru: Kesinlik ve Duyarlılık metriklerinin harmonik ortalaması
        "F1_Measure": f1_score(y_true, y_pred, zero_division=0),

        # Cohen's Kappa: Şans eseri oluşan doğru tahminleri cezalandıran, Stanford'ın MURA
        # yarışmasında ana sıralama (leaderboard) ölçütü olarak kullandığı en kritik metrik.
        "Cohen_Kappa": cohen_kappa_score(y_true, y_pred),

        # ROC-AUC: Alıcı İşletim Karakteristiği Eğrisi Altında Kalan Alan (Modelin sınıfları ayırma gücü)
        "ROC_AUC": roc_auc_score(y_true, y_scores),

        # PR-AUC (uAP): Kesinlik-Duyarlılık Eğrisi Altında Kalan Alan. Dengesiz veri setlerinde
        # ROC-AUC'ye kıyasla model başarısını çok daha objektif yansıtır.
        "PR_AUC_uAP": average_precision_score(y_true, y_scores), # Uninterpolated Average Precision

        # Hata Metrikleri: Tahmin edilen olasılıklar ile gerçek etiketler (0 veya 1) arasındaki mutlak ve karesel sapmalar
        "MAE": mean_absolute_error(y_true, y_scores),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_scores))
    }
    return metrikler

hücre 5 sözde kod

In [6]:
###Hücre 6: Ana Eğitim, Doğrulama ve Zaman Kayıt Döngüsü###

import time
import os
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve

# ==============================================================================
# EĞİTİM DÖNGÜSÜ HAZIRLIKLARI VE İZLEME DEĞİŞKENLERİ
# ==============================================================================
best_val_loss = float('inf') # En iyi modelin kaydedilmesi için başlangıç eşiği
patience_counter = 0         # Erken durdurma (Early Stopping) için sayaç
egitim_gecmisi = []          # Her epok sonunda elde edilen metriklerin toplanacağı liste

# --- SCHEDULER (ÖĞRENME ORANI PLANLAYICI) ---
# Modelin doğrulama kaybı (val loss) 3 epok boyunca iyileşmezse, öğrenme oranını
# yarıya (%50) düşürerek (factor=0.5) daha ince ayar yapmasını sağlar.
# "min" modu, hedefin kaybı küçültmek olduğunu belirtir.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print(f"[{CONFIG['experiment_name']}] Eğitim Başlıyor...")
toplam_baslangic_zamani = time.time()

# ==============================================================================
# ANA EPOK DÖNGÜSÜ
# ==============================================================================
for epoch in range(CONFIG["num_epochs"]):
    epoch_baslangic_zamani = time.time()

    # --- 1. EĞİTİM (TRAINING) FAZI ---
    model.train() # Modeli eğitim moduna alır (Dropout ve BatchNorm katmanları aktifleşir)
    train_loss = 0.0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['num_epochs']} - Eğitim")
    for inputs, labels in loop:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad() # Önceki iterasyondan kalan gradyanları sıfırla

        # İleri Yönlü Yayılım (Forward Pass) ve MixUp Veri Artırımı Kontrolü
        if CONFIG.get("use_mixup", False):
            inputs, targets_a, targets_b, lam = mixup_data(inputs, labels)
            outputs = model(inputs)
            loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        else:
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        # Geri Yayılım (Backward Pass) ve Ağırlık Güncellemesi
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)

        # İzleme çubuğuna (tqdm) farklılaştırılmış öğrenme oranlarının anlık yansıtılması
        lr_backbone = optimizer.param_groups[0]['lr']
        lr_head = optimizer.param_groups[-1]['lr']
        loop.set_postfix(loss=loss.item(), lr_govde=lr_backbone, lr_baslik=lr_head)

    # Epok sonu ortalama eğitim kaybının hesaplanması
    epoch_train_loss = train_loss / len(train_loader.dataset)

    # --- 2. DOĞRULAMA (VALIDATION) FAZI ---
    model.eval() # Modeli doğrulama moduna alır (Ağırlık güncellemeleri durur)
    val_loss = 0.0
    y_true, y_pred, y_scores = [], [], []
    toplam_cikarim_suresi = 0.0
    ornek_sayisi = 0

    with torch.no_grad(): # Hafıza tasarrufu için gradyan hesaplamasını tamamen kapat
        for inputs, labels in tqdm(val_loader, desc="Doğrulama"):
            inputs, labels = inputs.to(device), labels.to(device)

            # Çıkarım süresi (Inference Time) ölçümü (Klinik hız analizi için)
            start_infer = time.time()
            outputs = model(inputs)
            end_infer = time.time()

            toplam_cikarim_suresi += (end_infer - start_infer)
            ornek_sayisi += inputs.size(0)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)

            # Model çıktılarının olasılıklara (softmax) ve kesin sınıflara (argmax) dönüştürülmesi
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)

            # İstatistiksel hesaplamalar için değerlerin CPU'ya aktarılıp listelerde toplanması
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            y_scores.extend(probs[:, 1].cpu().numpy())

    # Epok sonu ortalama doğrulama kaybı ve milisaniye cinsinden çıkarım hızı
    epoch_val_loss = val_loss / len(val_loader.dataset)
    ms_per_step = (toplam_cikarim_suresi / ornek_sayisi) * 1000

    # --- 3. DİNAMİK OPTİMİZASYON VE KAYIT İŞLEMLERİ ---

    # Her epoch sonunda doğrulama kaybını scheduler'a bildirerek LR'nin ayarlanması
    scheduler.step(epoch_val_loss)

    # Hücre 5'teki fonksiyon çağrılarak tüm literatür metriklerinin hesaplanması
    metrikler = kapsamli_metrikleri_hesapla(y_true, y_pred, y_scores)

    epoch_bitis_zamani = time.time()
    epoch_suresi_sn = epoch_bitis_zamani - epoch_baslangic_zamani

    # Konsol Çıktıları
    # Not: current_lr değişkeni önceki kodlardan kalan bir kalıntı olabilir,
    # lr_head veya lr_backbone kullanımı daha doğru olurdu, ancak kod değiştirilmedi.
    current_lr = optimizer.param_groups[-1]['lr'] # Mevcut başlık öğrenme oranını okur
    print(f"\n--- Epoch {epoch+1} Sonuçları ---")
    print(f"Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Süre: {epoch_suresi_sn:.2f} sn | LR: {current_lr:.6f}")
    print(f"Accuracy: {metrikler['Accuracy']:.4f} | F1-Measure: {metrikler['F1_Measure']:.4f} | Kappa: {metrikler['Cohen_Kappa']:.4f}")
    print(f"PR-AUC (uAP): {metrikler['PR_AUC_uAP']:.4f} | ROC-AUC: {metrikler['ROC_AUC']:.4f}")
    print(f"Specificity: {metrikler['Specificity']:.4f} | Inference Time: {ms_per_step:.2f} ms/image")

    # Epok sonuçlarının genel listeye (Pandas DataFrame altyapısı) eklenmesi
    metrikler["Epoch"] = epoch + 1
    metrikler["Train_Loss"] = epoch_train_loss
    metrikler["Val_Loss"] = epoch_val_loss
    metrikler["Inference_Time_ms"] = ms_per_step
    metrikler["Epoch_Suresi_sn"] = epoch_suresi_sn
    metrikler["Learning_Rate"] = current_lr
    egitim_gecmisi.append(metrikler)

    # ==========================================================================
    # ERKEN DURDURMA (EARLY STOPPING) VE MODEL KAYDETME (CHECKPOINTING)
    # ==========================================================================
    # Aşırı öğrenmeyi (Overfitting) engellemek için, modelin ezberlemeden
    # genellenebilir örüntüler öğrendiği o altın "tepe noktası" diske kaydedilir.
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), f"/content/best_model_{CONFIG['model_architecture']}.pth")
    else:
        patience_counter += 1
        if patience_counter >= CONFIG["early_stopping_patience"]:
            print(f"\nErken Durdurma tetiklendi! {CONFIG['early_stopping_patience']} epoch boyunca Val Loss iyileşmedi.")
            break

toplam_bitis_zamani = time.time()
toplam_sure_dk = (toplam_bitis_zamani - toplam_baslangic_zamani) / 60
print(f"\nToplam Eğitim Süresi: {toplam_sure_dk:.2f} dakika.")

# ==============================================================================
# ÇIKTILARI, GRAFİKLERİ VE HİPERPARAMETRELERİ KAYDETME BÖLÜMÜ
# ==============================================================================
print("\nSonuçlar, grafikler ve hiperparametreler Google Drive'a kaydediliyor...")

# Drive üzerinde bu deneye özel benzersiz bir sonuç klasörü oluşturulur
grafik_klasoru = f"/content/drive/MyDrive/Doktora/Verisetleri_tik/deneyler/{CONFIG['experiment_name']}_Sonuclar"
os.makedirs(grafik_klasoru, exist_ok=True)

# 1. Epok epok tüm eğitim geçmişi sayısal formatta (CSV) kaydedilir
df_sonuclar = pd.DataFrame(egitim_gecmisi)
df_sonuclar.to_csv(os.path.join(grafik_klasoru, "Egitim_Metrikleri.csv"), index=False)

# 2. Deneyin birebir tekrar üretilebilmesi için CONFIG ayarları JSON olarak kaydedilir
kayit_verisi = CONFIG.copy()
kayit_verisi["Zaman_Bilgileri"] = {
    "Toplam_Egitim_Suresi_Dakika": round(toplam_sure_dk, 2),
    "Epoch_Sureleri_Saniye": [round(m["Epoch_Suresi_sn"], 2) for m in egitim_gecmisi]
}
kayit_verisi["Learning_Rate_Gecmisi"] = [m["Learning_Rate"] for m in egitim_gecmisi]

hiperparametre_dosyasi = os.path.join(grafik_klasoru, "Hiperparametreler.json")
with open(hiperparametre_dosyasi, "w", encoding="utf-8") as f:
    json.dump(kayit_verisi, f, indent=4, ensure_ascii=False)

# --- MATPLOTLIB İLE GÖRSELLEŞTİRME (KAYIP VE DOĞRULUK EĞRİLERİ) ---
# 1. Eğitim ve Doğrulama Kaybı (Loss) Eğrisi
plt.figure(figsize=(10, 5))
plt.plot(df_sonuclar['Epoch'], df_sonuclar['Train_Loss'], label='Training Loss', marker='o', markersize=4)
plt.plot(df_sonuclar['Epoch'], df_sonuclar['Val_Loss'], label='Validation Loss', marker='o', markersize=4)
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(grafik_klasoru, "1_Training_Validation_Loss_Curve.png"), dpi=300)
plt.close()

# 2. Accuracy Eğrisi
plt.figure(figsize=(10, 5))
plt.plot(df_sonuclar['Epoch'], df_sonuclar['Accuracy'], label='Accuracy Curve', color='green', marker='o', markersize=4)
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(grafik_klasoru, "2_Accuracy_Curve.png"), dpi=300)
plt.close()

# ==============================================================================
# KRİTİK DÜZELTME: EN İYİ MODELİ GERİ YÜKLEME VE YENİDEN TAHMİN (BEST MODEL INFERENCE)
# ==============================================================================
# Eğer eğitim "Early Stopping" ile kesilirse, o an RAM'de bulunan model aslında
# ezberlemeye başlamış olan "kötü" modeldir. Nihai grafikleri (Confusion Matrix vb.)
# çizerken yanıltıcı sonuçlar almamak için, daha önce diske kaydedilen o "altın"
# ağırlıklar (best_model) geri yüklenir ve validasyon seti üzerinden tekrar test edilir.
print("\nGrafikler için 'En İyi Model (Best Model)' ağırlıkları geri yükleniyor...")
model.load_state_dict(torch.load(f"/content/best_model_{CONFIG['model_architecture']}.pth"))
model.eval()

y_true_best, y_pred_best, y_scores_best = [], [], []

# Sadece doğrulama seti üzerinden en iyi ağırlıklarla tekrar tahmin (inference) alıyoruz
with torch.no_grad():
    for inputs, labels in tqdm(val_loader, desc="Best Model Değerlendirmesi"):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)

        y_true_best.extend(labels.cpu().numpy())
        y_pred_best.extend(preds.cpu().numpy())
        y_scores_best.extend(probs[:, 1].cpu().numpy())

# --- NİHAİ BAŞARIM GRAFİKLERİ (BEST MODEL ÜZERİNDEN) ---
# 3. Karmaşıklık Matrisi (Heatmap)
cm_default = confusion_matrix(y_true_best, y_pred_best)
tn, fp, fn, tp = cm_default.ravel() if cm_default.size == 4 else (0, 0, 0, 0)
cm_custom = np.array([[tp, fp],
                      [fn, tn]])
plt.figure(figsize=(8, 6))
sns.heatmap(cm_custom, annot=True, fmt='d', cmap='Wistia', cbar=False,
            annot_kws={"size": 16, "weight": "bold"},
            xticklabels=['Actual Positive (1)', 'Actual Negative (0)'],
            yticklabels=['Predicted Positive (1)', 'Predicted Negative (0)'],
            linewidths=1, linecolor='black')
plt.title('Confusion Matrix', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Actual Values', fontsize=14, fontweight='bold', labelpad=10)
plt.ylabel('Predicted Values', fontsize=14, fontweight='bold', labelpad=10)
plt.yticks(rotation=90, va="center")
plt.tight_layout()
plt.savefig(os.path.join(grafik_klasoru, "3_Confusion_Matrix.png"), dpi=300)
plt.close()

# 4. ROC Eğrisi
fpr, tpr, _ = roc_curve(y_true_best, y_scores_best)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(grafik_klasoru, "4_ROC_Curve.png"), dpi=300)
plt.close()

# 5. Kesinlik-Duyarlılık (Precision-Recall) Eğrisi
precision, recall, _ = precision_recall_curve(y_true_best, y_scores_best)
plt.figure(figsize=(7, 6))
plt.plot(recall, precision, color='purple', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(grafik_klasoru, "5_PR_Curve.png"), dpi=300)
plt.close()

print(f"Tüm grafikler başarıyla '{grafik_klasoru}' klasörüne kaydedildi.")

[Exp 2.13: MURA and Modern CNN(regnet_y_3_2gf)] Eğitim Başlıyor...


Doğrulama: 100%|██████████| 100/100 [00:02<00:00, 38.75it/s]



--- Epoch 1 Sonuçları ---
Train Loss: 0.6100 | Val Loss: 0.5484 | Süre: 76.83 sn | LR: 0.000050
Accuracy: 0.7435 | F1-Measure: 0.6947 | Kappa: 0.4809
PR-AUC (uAP): 0.8192 | ROC-AUC: 0.8095
Specificity: 0.8662 | Inference Time: 0.62 ms/image


Doğrulama: 100%|██████████| 100/100 [00:02<00:00, 37.86it/s]



--- Epoch 2 Sonuçları ---
Train Loss: 0.5194 | Val Loss: 0.4969 | Süre: 75.38 sn | LR: 0.000050
Accuracy: 0.7645 | F1-Measure: 0.7302 | Kappa: 0.5246
PR-AUC (uAP): 0.8521 | ROC-AUC: 0.8412
Specificity: 0.8548 | Inference Time: 0.66 ms/image


...


--- Epoch 19 Sonuçları ---
Train Loss: 0.3856 | Val Loss: 0.4723 | Süre: 75.07 sn | LR: 0.000013
Accuracy: 0.8145 | F1-Measure: 0.7932 | Kappa: 0.6264
PR-AUC (uAP): 0.8825 | ROC-AUC: 0.8736
Specificity: 0.8800 | Inference Time: 0.70 ms/image


Doğrulama: 100%|██████████| 100/100 [00:02<00:00, 36.87it/s]



--- Epoch 20 Sonuçları ---
Train Loss: 0.3835 | Val Loss: 0.4620 | Süre: 74.88 sn | LR: 0.000006
Accuracy: 0.8145 | F1-Measure: 0.7971 | Kappa: 0.6270
PR-AUC (uAP): 0.8819 | ROC-AUC: 0.8729
Specificity: 0.8632 | Inference Time: 0.68 ms/image

Erken Durdurma tetiklendi! 12 epoch boyunca Val Loss iyileşmedi.

Toplam Eğitim Süresi: 25.05 dakika.

Sonuçlar, grafikler ve hiperparametreler Google Drive'a kaydediliyor...

Grafikler için 'En İyi Model (Best Model)' ağırlıkları geri yükleniyor...


Best Model Değerlendirmesi: 100%|██████████| 100/100 [00:02<00:00, 37.10it/s]


Tüm grafikler başarıyla '/content/drive/MyDrive/Doktora/Verisetleri_tik/deneyler/Exp 2.13: MURA and Modern CNN(regnet_y_3_2gf)_Sonuclar' klasörüne kaydedildi.


hücre 6 sözde kod